# Notebook 06: Additional Experiments for IEEE Access Reviewer Response
Addresses reviewer comments R1-2, R1-3, R1-7, R1-8, R1-10, R1-12, R2-2

Experiments:
1. Simpler baselines comparison (R1-8)
2. MC-dropout on attention-LSTM (R1-2)
3. Attention perturbation/ablation (R1-3)
4. Hyperparameter sensitivity analysis (R1-7)
5. Inference latency measurement (R1-10)
6. Window sensitivity analysis (R1-12)
7. Extended seed analysis with 16 seeds (R1-6)

In [ ]:
import numpy as np
import pandas as pd
import pickle
import time
import json
import warnings
import os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from scipy.stats import pearsonr

results_dir = Path('../results')
tables_dir = results_dir / 'tables'
figures_dir = results_dir / 'figures'

# Load data
print("Loading data...")
X_train_full = np.load(results_dir / 'X_train.npy')
X_test = np.load(results_dir / 'X_test.npy')
y_train_full = np.load(results_dir / 'y_train.npy')
y_test = np.load(results_dir / 'y_test.npy')

with open(results_dir / 'feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

with open(results_dir / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print(f"X_train: {X_train_full.shape}, X_test: {X_test.shape}")
print(f"Features: {feature_names}")

# Create validation split
val_size = int(0.2 * len(X_train_full))
X_val = X_train_full[-val_size:]
y_val = y_train_full[-val_size:]
X_train = X_train_full[:-val_size]
y_train = y_train_full[:-val_size]

all_results = {}

## EXPERIMENT 1: SIMPLER BASELINES (R1-8)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 1: SIMPLER BASELINES COMPARISON (R1-8)")
print("="*70)

# Flatten 3D data for sklearn models
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)

# Use subsample for training SVR (too slow on full data)
subsample_size = 50000
np.random.seed(42)
sub_idx = np.random.choice(len(X_train_flat), subsample_size, replace=False)
X_train_sub = X_train_flat[sub_idx]
y_train_sub = y_train[sub_idx]

baseline_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'SVR (RBF)': SVR(kernel='rbf', C=1.0, epsilon=0.01),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
}

baseline_results = []

for name, model in baseline_models.items():
    print(f"\nTraining {name}...")
    start_time = time.time()

    # Use subsample for SVR, full data for others
    if 'SVR' in name:
        model.fit(X_train_sub, y_train_sub)
    elif 'Random Forest' in name or 'Gradient Boosting' in name:
        # Use larger subsample for tree models
        tree_sub = min(200000, len(X_train_flat))
        tree_idx = np.random.choice(len(X_train_flat), tree_sub, replace=False)
        model.fit(X_train_flat[tree_idx], y_train[tree_idx])
    else:
        model.fit(X_train_flat, y_train)

    train_time = time.time() - start_time

    # Predict
    start_time = time.time()
    y_pred = model.predict(X_test_flat)
    inference_time = time.time() - start_time

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    baseline_results.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Training_Time_s': train_time,
        'Inference_Time_s': inference_time,
        'Parameters': 'N/A'
    })

    print(f"  RMSE: {rmse:.6f}, MAE: {mae:.6f}, R²: {r2:.4f}")
    print(f"  Train time: {train_time:.1f}s, Inference time: {inference_time:.2f}s")

# Add LSTM results from existing data
baseline_results.append({
    'Model': 'Baseline LSTM (best seed)',
    'RMSE': 0.0185,
    'MAE': 0.0090,
    'R2': 0.3266,
    'Training_Time_s': 'N/A',
    'Inference_Time_s': 'N/A',
    'Parameters': '2,181'
})
baseline_results.append({
    'Model': 'Attention-LSTM (best seed)',
    'RMSE': 0.0094,
    'MAE': 0.0030,
    'R2': 0.8270,
    'Training_Time_s': 'N/A',
    'Inference_Time_s': 'N/A',
    'Parameters': '2,621'
})

baseline_df = pd.DataFrame(baseline_results)
baseline_df.to_csv(tables_dir / 'baseline_comparison_full.csv', index=False)
print(f"\n✅ Baselines comparison saved to {tables_dir / 'baseline_comparison_full.csv'}")
print(baseline_df.to_string(index=False))

all_results['baselines'] = baseline_results

## EXPERIMENT 2: MC-DROPOUT ON ATTENTION-LSTM (R1-2)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 2: MC-DROPOUT ON ATTENTION-LSTM (R1-2)")
print("="*70)

import tensorflow as tf
from tensorflow import keras

# Load attention model
print("Loading attention-LSTM model...")
from tensorflow.keras.layers import Layer

class AttentionLayer(Layer):
    def __init__(self, units=20, return_sequences=True, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.units = units
        self.return_sequences = return_sequences

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], self.units),
                                  initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(self.units,),
                                  initializer='zeros', trainable=True)
        self.u = self.add_weight(name='attention_vector', shape=(self.units,),
                                  initializer='glorot_uniform', trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        attention_weights = tf.nn.softmax(tf.tensordot(score, self.u, axes=1), axis=1)
        attention_weights = tf.expand_dims(attention_weights, axis=-1)
        context_vector = tf.reduce_sum(inputs * attention_weights, axis=1)
        if self.return_sequences:
            return context_vector, tf.squeeze(attention_weights, axis=-1)
        return context_vector

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'return_sequences': self.return_sequences})
        return config

# Build fresh attention model with dropout for MC-dropout
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout

def build_attention_lstm_with_dropout(input_shape, dropout_rate=0.2):
    inputs = Input(shape=input_shape)
    lstm_output = LSTM(20, return_sequences=True)(inputs)
    dropout_layer = Dropout(dropout_rate)(lstm_output)
    context_vector, attention_weights = AttentionLayer(units=20)(dropout_layer)
    outputs = Dense(1)(context_vector)
    model = Model(inputs=inputs, outputs=outputs)
    attention_model = Model(inputs=inputs, outputs=attention_weights)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model, attention_model

# Try to load existing model weights, or train new one
attention_model_path = results_dir / 'models' / 'attention_lstm_final.h5'

# We need to train a NEW attention model with dropout layer for MC-dropout
print("Training attention-LSTM with dropout for MC-dropout inference...")
np.random.seed(42)
tf.random.set_seed(42)

attn_model_mc, attn_weights_model = build_attention_lstm_with_dropout(
    (X_train.shape[1], X_train.shape[2]), dropout_rate=0.2
)

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

attn_model_mc.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

# Standard prediction
y_pred_attn = attn_model_mc.predict(X_test, verbose=0).flatten()
attn_rmse = np.sqrt(mean_squared_error(y_test, y_pred_attn))
attn_mae = mean_absolute_error(y_test, y_pred_attn)
attn_r2 = r2_score(y_test, y_pred_attn)
print(f"\nAttention-LSTM (with dropout) standard prediction:")
print(f"  RMSE: {attn_rmse:.6f}, MAE: {attn_mae:.6f}, R²: {attn_r2:.4f}")

# MC-dropout inference
print("\nPerforming MC-dropout inference (100 passes)...")
n_mc = 100
mc_predictions = []
for i in range(n_mc):
    y_mc = attn_model_mc(X_test, training=True).numpy().flatten()
    mc_predictions.append(y_mc)
    if (i+1) % 20 == 0:
        print(f"  Completed {i+1}/{n_mc} passes...")

mc_predictions = np.array(mc_predictions)
y_mc_mean = mc_predictions.mean(axis=0)
y_mc_std = mc_predictions.std(axis=0)

# Metrics
mc_rmse = np.sqrt(mean_squared_error(y_test, y_mc_mean))
mc_mae = mean_absolute_error(y_test, y_mc_mean)
mc_r2 = r2_score(y_test, y_mc_mean)
mc_uncertainty_mean = y_mc_std.mean()
mc_uncertainty_std = y_mc_std.std()

# Uncertainty-error correlation
abs_errors = np.abs(y_test - y_mc_mean)
ue_corr, ue_p = pearsonr(y_mc_std, abs_errors)

print(f"\nAttention-LSTM MC-dropout results:")
print(f"  RMSE: {mc_rmse:.6f}, MAE: {mc_mae:.6f}, R²: {mc_r2:.4f}")
print(f"  Mean uncertainty (σ): {mc_uncertainty_mean:.6f} ± {mc_uncertainty_std:.6f}")
print(f"  Uncertainty-error correlation: r={ue_corr:.4f}, p={ue_p:.2e}")

# Compare with baseline MC-dropout
baseline_mc_std = np.load(results_dir / 'y_test_std_mc_dropout.npy')
baseline_mc_mean_unc = baseline_mc_std.mean()

print(f"\n--- Comparison ---")
print(f"  Baseline LSTM mean uncertainty: {baseline_mc_mean_unc:.6f}")
print(f"  Attention-LSTM mean uncertainty: {mc_uncertainty_mean:.6f}")
print(f"  Uncertainty reduction: {((baseline_mc_mean_unc - mc_uncertainty_mean)/baseline_mc_mean_unc*100):.1f}%")

# Save MC-dropout results
np.save(results_dir / 'y_test_mean_mc_dropout_attention.npy', y_mc_mean)
np.save(results_dir / 'y_test_std_mc_dropout_attention.npy', y_mc_std)

mc_dropout_results = {
    'Baseline_LSTM': {
        'mean_uncertainty': float(baseline_mc_mean_unc),
        'uncertainty_error_corr': 0.523,  # from paper
    },
    'Attention_LSTM': {
        'RMSE': float(mc_rmse),
        'MAE': float(mc_mae),
        'R2': float(mc_r2),
        'mean_uncertainty': float(mc_uncertainty_mean),
        'std_uncertainty': float(mc_uncertainty_std),
        'uncertainty_error_corr': float(ue_corr),
        'uncertainty_error_p': float(ue_p),
    }
}
all_results['mc_dropout'] = mc_dropout_results
print("✅ MC-dropout on attention-LSTM complete")

## EXPERIMENT 3: ATTENTION PERTURBATION/ABLATION (R1-3)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 3: ATTENTION PERTURBATION/ABLATION (R1-3)")
print("="*70)

# Load attention weights
attention_weights = np.load(results_dir / 'attention_weights.npy')

# Load original attention model for perturbation tests
attn_model_orig = keras.models.load_model(
    str(results_dir / 'models' / 'attention_lstm_final.h5'),
    custom_objects={'AttentionLayer': AttentionLayer}
)

# Test 1: Mask high-attention timesteps and measure degradation
print("\n--- Perturbation Test 1: Mask high-attention timesteps ---")
avg_attention = attention_weights.mean(axis=0)
timestep_importance_order = np.argsort(avg_attention)[::-1]  # Most important first

# Use a subsample for speed
n_perturb = min(50000, len(X_test))
perturb_idx = np.random.choice(len(X_test), n_perturb, replace=False)
X_perturb = X_test[perturb_idx].copy()
y_perturb = y_test[perturb_idx]

# Baseline (unperturbed)
y_base = attn_model_orig.predict(X_perturb, verbose=0).flatten()
base_mae = mean_absolute_error(y_perturb, y_base)
base_r2 = r2_score(y_perturb, y_base)

perturbation_results = [{'Masked_Timesteps': 'None (baseline)', 'MAE': base_mae, 'R2': base_r2, 'MAE_Increase_Pct': 0.0}]

# Progressively mask top-K attended timesteps (replace with training mean)
train_mean = X_train.mean(axis=0)  # (15, 6)

for k in range(1, 6):
    X_masked = X_perturb.copy()
    top_k = timestep_importance_order[:k]
    for t in top_k:
        X_masked[:, t, :] = train_mean[t, :]

    y_masked = attn_model_orig.predict(X_masked, verbose=0).flatten()
    masked_mae = mean_absolute_error(y_perturb, y_masked)
    masked_r2 = r2_score(y_perturb, y_masked)
    mae_increase = ((masked_mae - base_mae) / base_mae) * 100

    timestep_names = [f"t-{15-t}" for t in top_k]
    perturbation_results.append({
        'Masked_Timesteps': ', '.join(timestep_names),
        'MAE': masked_mae,
        'R2': masked_r2,
        'MAE_Increase_Pct': mae_increase
    })
    print(f"  Mask top-{k} ({', '.join(timestep_names)}): MAE={masked_mae:.6f} (+{mae_increase:.1f}%), R²={masked_r2:.4f}")

# Test 2: Mask LOW-attention timesteps (should have less impact)
print("\n--- Perturbation Test 2: Mask low-attention timesteps ---")
for k in range(1, 6):
    X_masked = X_perturb.copy()
    bottom_k = timestep_importance_order[-k:]
    for t in bottom_k:
        X_masked[:, t, :] = train_mean[t, :]

    y_masked = attn_model_orig.predict(X_masked, verbose=0).flatten()
    masked_mae = mean_absolute_error(y_perturb, y_masked)
    masked_r2 = r2_score(y_perturb, y_masked)
    mae_increase = ((masked_mae - base_mae) / base_mae) * 100

    timestep_names = [f"t-{15-t}" for t in bottom_k]
    perturbation_results.append({
        'Masked_Timesteps': f'Bottom-{k}: {", ".join(timestep_names)}',
        'MAE': masked_mae,
        'R2': masked_r2,
        'MAE_Increase_Pct': mae_increase
    })
    print(f"  Mask bottom-{k} ({', '.join(timestep_names)}): MAE={masked_mae:.6f} (+{mae_increase:.1f}%), R²={masked_r2:.4f}")

# Test 3: Random shuffling of timesteps within each sample
print("\n--- Perturbation Test 3: Random timestep shuffling ---")
X_shuffled = X_perturb.copy()
for i in range(len(X_shuffled)):
    perm = np.random.permutation(15)
    X_shuffled[i] = X_shuffled[i, perm, :]

y_shuffled = attn_model_orig.predict(X_shuffled, verbose=0).flatten()
shuffled_mae = mean_absolute_error(y_perturb, y_shuffled)
shuffled_r2 = r2_score(y_perturb, y_shuffled)
shuffle_increase = ((shuffled_mae - base_mae) / base_mae) * 100

perturbation_results.append({
    'Masked_Timesteps': 'Random shuffle all timesteps',
    'MAE': shuffled_mae,
    'R2': shuffled_r2,
    'MAE_Increase_Pct': shuffle_increase
})
print(f"  Random shuffle: MAE={shuffled_mae:.6f} (+{shuffle_increase:.1f}%), R²={shuffled_r2:.4f}")

perturbation_df = pd.DataFrame(perturbation_results)
perturbation_df.to_csv(tables_dir / 'attention_perturbation_results.csv', index=False)
print(f"\n✅ Perturbation results saved")
all_results['perturbation'] = perturbation_results

## EXPERIMENT 4: HYPERPARAMETER SENSITIVITY (R1-7)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 4: HYPERPARAMETER SENSITIVITY ANALYSIS (R1-7)")
print("="*70)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM as LSTM_Layer, Dense as Dense_Layer, Dropout as Dropout_Layer

# Use smaller subsample for faster hyperparameter search
hyp_sub = min(100000, len(X_train))
hyp_idx = np.random.choice(len(X_train), hyp_sub, replace=False)
X_hyp = X_train[hyp_idx]
y_hyp = y_train[hyp_idx]

hyperparam_results = []

# Test different configurations for BASELINE LSTM
configs = [
    {'lr': 0.001, 'dropout': 0.2, 'units': 20, 'label': 'Default (paper)'},
    {'lr': 0.0001, 'dropout': 0.2, 'units': 20, 'label': 'Lower LR (0.0001)'},
    {'lr': 0.01, 'dropout': 0.2, 'units': 20, 'label': 'Higher LR (0.01)'},
    {'lr': 0.001, 'dropout': 0.0, 'units': 20, 'label': 'No dropout'},
    {'lr': 0.001, 'dropout': 0.4, 'units': 20, 'label': 'Higher dropout (0.4)'},
    {'lr': 0.001, 'dropout': 0.2, 'units': 10, 'label': 'Fewer units (10)'},
    {'lr': 0.001, 'dropout': 0.2, 'units': 50, 'label': 'More units (50)'},
    {'lr': 0.001, 'dropout': 0.2, 'units': 100, 'label': 'Many units (100)'},
]

SEEDS_HYPER = [42, 123, 456, 789]

for config in configs:
    print(f"\nConfig: {config['label']}")
    seed_metrics = {'rmse': [], 'mae': [], 'r2': []}
    converged_count = 0

    for seed in SEEDS_HYPER:
        np.random.seed(seed)
        tf.random.set_seed(seed)

        model = Sequential([
            LSTM_Layer(config['units'], input_shape=(X_hyp.shape[1], X_hyp.shape[2])),
            Dropout_Layer(config['dropout']),
            Dense_Layer(1)
        ])
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
                      loss='mse', metrics=['mae'])

        early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        model.fit(X_hyp, y_hyp, validation_data=(X_val, y_val),
                  epochs=50, batch_size=256, callbacks=[early_stop], verbose=0)

        y_pred = model.predict(X_test, verbose=0).flatten()
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        seed_metrics['rmse'].append(rmse)
        seed_metrics['mae'].append(mae)
        seed_metrics['r2'].append(r2)

        if r2 > -1.0:  # Consider converged if R² > -1
            converged_count += 1

    result = {
        'Configuration': config['label'],
        'LR': config['lr'],
        'Dropout': config['dropout'],
        'Units': config['units'],
        'Convergence_Rate': f"{converged_count}/{len(SEEDS_HYPER)}",
        'Mean_RMSE': np.mean(seed_metrics['rmse']),
        'Std_RMSE': np.std(seed_metrics['rmse']),
        'Mean_MAE': np.mean(seed_metrics['mae']),
        'Std_MAE': np.std(seed_metrics['mae']),
        'Mean_R2': np.mean(seed_metrics['r2']),
        'Std_R2': np.std(seed_metrics['r2']),
        'Best_R2': max(seed_metrics['r2']),
        'Worst_R2': min(seed_metrics['r2']),
    }
    hyperparam_results.append(result)
    print(f"  Conv: {converged_count}/{len(SEEDS_HYPER)}, "
          f"MAE: {result['Mean_MAE']:.4f}±{result['Std_MAE']:.4f}, "
          f"R²: {result['Mean_R2']:.4f}±{result['Std_R2']:.4f}")

hyperparam_df = pd.DataFrame(hyperparam_results)
hyperparam_df.to_csv(tables_dir / 'hyperparameter_sensitivity.csv', index=False)
print(f"\n✅ Hyperparameter sensitivity saved")
all_results['hyperparameters'] = hyperparam_results

## EXPERIMENT 5: INFERENCE LATENCY (R1-10)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 5: INFERENCE LATENCY ANALYSIS (R1-10)")
print("="*70)

# Load both models
baseline_model = keras.models.load_model(str(results_dir / 'models' / 'baseline_lstm_final.h5'))
attention_model_full = keras.models.load_model(
    str(results_dir / 'models' / 'attention_lstm_final.h5'),
    custom_objects={'AttentionLayer': AttentionLayer}
)

latency_results = []

# Single sample latency (most relevant for real-time)
batch_sizes = [1, 10, 100, 1000]
n_repeats = 50

for bs in batch_sizes:
    X_batch = X_test[:bs]

    # Baseline LSTM
    times_baseline = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        _ = baseline_model.predict(X_batch, verbose=0)
        times_baseline.append(time.perf_counter() - start)

    # Attention-LSTM
    times_attention = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        _ = attention_model_full.predict(X_batch, verbose=0)
        times_attention.append(time.perf_counter() - start)

    # MC-dropout (100 passes, single sample)
    if bs <= 10:
        times_mc = []
        for _ in range(min(10, n_repeats)):
            start = time.perf_counter()
            for _ in range(100):
                _ = attn_model_mc(X_batch, training=True)
            times_mc.append(time.perf_counter() - start)
        mc_mean = np.mean(times_mc) * 1000
    else:
        mc_mean = float('nan')

    # SHAP (estimated from 1 sample)
    if bs == 1:
        from sklearn.linear_model import LinearRegression as LR_temp
        # Rough estimate: SHAP takes ~0.1-0.5s per sample based on notebook
        shap_estimated = 125.0  # ms per sample (from 20 samples in ~2.5s)
    else:
        shap_estimated = float('nan')

    result = {
        'Batch_Size': bs,
        'Baseline_LSTM_ms': np.mean(times_baseline) * 1000,
        'Baseline_LSTM_std_ms': np.std(times_baseline) * 1000,
        'Attention_LSTM_ms': np.mean(times_attention) * 1000,
        'Attention_LSTM_std_ms': np.std(times_attention) * 1000,
        'MC_Dropout_100passes_ms': mc_mean,
        'Per_sample_baseline_ms': np.mean(times_baseline) * 1000 / bs,
        'Per_sample_attention_ms': np.mean(times_attention) * 1000 / bs,
    }
    latency_results.append(result)
    print(f"  Batch={bs}: Baseline={result['Baseline_LSTM_ms']:.2f}ms, "
          f"Attention={result['Attention_LSTM_ms']:.2f}ms, "
          f"Per-sample: {result['Per_sample_attention_ms']:.3f}ms")

latency_df = pd.DataFrame(latency_results)
latency_df.to_csv(tables_dir / 'inference_latency.csv', index=False)
print(f"\n✅ Inference latency results saved")
all_results['latency'] = latency_results

## EXPERIMENT 6: WINDOW SENSITIVITY (R1-12)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 6: WINDOW SENSITIVITY ANALYSIS (R1-12)")
print("="*70)

# Reload raw data to create different window sizes
print("Loading raw data for window analysis...")
data_path = Path('../github_repository/data/data.csv')
if not data_path.exists():
    data_path = Path('../../data.csv')

df_raw = pd.read_csv(str(data_path), encoding='latin-1')
feature_cols = ['humidity', 'temperature', 'humiditysol', 'temperaturesol', 'co2', 'lumière']
df_clean = df_raw[feature_cols].dropna()

from sklearn.preprocessing import MinMaxScaler
scaler_w = MinMaxScaler()
scaled = scaler_w.fit_transform(df_clean.values)

window_sizes = [5, 10, 15, 20, 30]
window_results = []
SEEDS_WINDOW = [42, 123, 456]

for ws in window_sizes:
    print(f"\nWindow size: {ws} timesteps ({ws*5} minutes)")

    # Create sequences
    X_w, y_w = [], []
    for i in range(ws, len(scaled)):
        X_w.append(scaled[i-ws:i, :])
        y_w.append(scaled[i, 2])  # humiditysol
    X_w = np.array(X_w)
    y_w = np.array(y_w)

    train_size = int(len(X_w) * 0.8)
    X_train_w = X_w[:train_size]
    y_train_w = y_w[:train_size]
    X_test_w = X_w[train_size:]
    y_test_w = y_w[train_size:]

    # Use subsample for training speed
    sub_w = min(100000, len(X_train_w))
    sub_idx_w = np.random.choice(len(X_train_w), sub_w, replace=False)
    val_w = int(0.2 * sub_w)

    seed_metrics = {'rmse': [], 'mae': [], 'r2': [], 'converged': 0}

    for seed in SEEDS_WINDOW:
        np.random.seed(seed)
        tf.random.set_seed(seed)

        # Build attention-LSTM
        inputs = Input(shape=(ws, 6))
        lstm_out = LSTM(20, return_sequences=True)(inputs)
        context, attn_w = AttentionLayer(units=20)(lstm_out)
        outputs = Dense(1)(context)
        model_w = Model(inputs=inputs, outputs=outputs)
        model_w.compile(optimizer='adam', loss='mse', metrics=['mae'])

        X_tr = X_train_w[sub_idx_w[:sub_w-val_w]]
        y_tr = y_train_w[sub_idx_w[:sub_w-val_w]]
        X_vl = X_train_w[sub_idx_w[sub_w-val_w:]]
        y_vl = y_train_w[sub_idx_w[sub_w-val_w:]]

        es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        model_w.fit(X_tr, y_tr, validation_data=(X_vl, y_vl), epochs=50, batch_size=256,
                    callbacks=[es], verbose=0)

        y_pred_w = model_w.predict(X_test_w, verbose=0).flatten()
        rmse_w = np.sqrt(mean_squared_error(y_test_w, y_pred_w))
        mae_w = mean_absolute_error(y_test_w, y_pred_w)
        r2_w = r2_score(y_test_w, y_pred_w)

        seed_metrics['rmse'].append(rmse_w)
        seed_metrics['mae'].append(mae_w)
        seed_metrics['r2'].append(r2_w)
        if r2_w > 0:
            seed_metrics['converged'] += 1

    result = {
        'Window_Size': ws,
        'Duration_Minutes': ws * 5,
        'Convergence_Rate': f"{seed_metrics['converged']}/{len(SEEDS_WINDOW)}",
        'Mean_RMSE': np.mean(seed_metrics['rmse']),
        'Std_RMSE': np.std(seed_metrics['rmse']),
        'Mean_MAE': np.mean(seed_metrics['mae']),
        'Std_MAE': np.std(seed_metrics['mae']),
        'Mean_R2': np.mean(seed_metrics['r2']),
        'Std_R2': np.std(seed_metrics['r2']),
    }
    window_results.append(result)
    print(f"  Conv: {seed_metrics['converged']}/{len(SEEDS_WINDOW)}, "
          f"MAE: {result['Mean_MAE']:.4f}±{result['Std_MAE']:.4f}, "
          f"R²: {result['Mean_R2']:.4f}±{result['Std_R2']:.4f}")

window_df = pd.DataFrame(window_results)
window_df.to_csv(tables_dir / 'window_sensitivity.csv', index=False)
print(f"\n✅ Window sensitivity results saved")
all_results['window_sensitivity'] = window_results

## EXPERIMENT 7: EXTENDED SEED ANALYSIS (R1-6)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 7: EXTENDED SEED ANALYSIS (R1-6)")
print("="*70)

SEEDS_EXTENDED = [42, 123, 456, 789, 1011, 1213, 1415, 1617,
                  2020, 3030, 4040, 5050, 6060, 7070, 8080, 9090]

# Use subsample for speed
seed_sub = min(100000, len(X_train))
seed_idx = np.random.choice(len(X_train), seed_sub, replace=False)
val_s = int(0.2 * seed_sub)

extended_seed_results = {'baseline': [], 'attention': []}

print(f"\nTraining {len(SEEDS_EXTENDED)} seeds for both architectures...")

for seed in SEEDS_EXTENDED:
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Baseline LSTM
    model_b = Sequential([
        LSTM_Layer(20, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout_Layer(0.2),
        Dense_Layer(1)
    ])
    model_b.compile(optimizer='adam', loss='mse', metrics=['mae'])
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model_b.fit(X_train[seed_idx[:seed_sub-val_s]], y_train[seed_idx[:seed_sub-val_s]],
                validation_data=(X_train[seed_idx[seed_sub-val_s:]], y_train[seed_idx[seed_sub-val_s:]]),
                epochs=50, batch_size=256, callbacks=[es], verbose=0)
    y_pred_b = model_b.predict(X_test, verbose=0).flatten()

    extended_seed_results['baseline'].append({
        'Seed': seed,
        'RMSE': float(np.sqrt(mean_squared_error(y_test, y_pred_b))),
        'MAE': float(mean_absolute_error(y_test, y_pred_b)),
        'R2': float(r2_score(y_test, y_pred_b)),
    })

    # Attention LSTM
    np.random.seed(seed)
    tf.random.set_seed(seed)

    inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))
    lstm_out = LSTM(20, return_sequences=True)(inputs)
    context, _ = AttentionLayer(units=20)(lstm_out)
    outputs = Dense(1)(context)
    model_a = Model(inputs=inputs, outputs=outputs)
    model_a.compile(optimizer='adam', loss='mse', metrics=['mae'])

    model_a.fit(X_train[seed_idx[:seed_sub-val_s]], y_train[seed_idx[:seed_sub-val_s]],
                validation_data=(X_train[seed_idx[seed_sub-val_s:]], y_train[seed_idx[seed_sub-val_s:]]),
                epochs=50, batch_size=256, callbacks=[es], verbose=0)
    y_pred_a = model_a.predict(X_test, verbose=0).flatten()

    extended_seed_results['attention'].append({
        'Seed': seed,
        'RMSE': float(np.sqrt(mean_squared_error(y_test, y_pred_a))),
        'MAE': float(mean_absolute_error(y_test, y_pred_a)),
        'R2': float(r2_score(y_test, y_pred_a)),
    })

    b_r2 = extended_seed_results['baseline'][-1]['R2']
    a_r2 = extended_seed_results['attention'][-1]['R2']
    print(f"  Seed {seed}: Baseline R²={b_r2:.4f}, Attention R²={a_r2:.4f}")

# Save extended results
ext_baseline_df = pd.DataFrame(extended_seed_results['baseline'])
ext_attention_df = pd.DataFrame(extended_seed_results['attention'])
ext_baseline_df.to_csv(tables_dir / 'extended_seed_baseline.csv', index=False)
ext_attention_df.to_csv(tables_dir / 'extended_seed_attention.csv', index=False)

# Summary
b_converged = sum(1 for r in extended_seed_results['baseline'] if r['R2'] > -1.0)
a_converged = sum(1 for r in extended_seed_results['attention'] if r['R2'] > 0)
b_r2s = [r['R2'] for r in extended_seed_results['baseline']]
a_r2s = [r['R2'] for r in extended_seed_results['attention']]

print(f"\n--- Extended Seed Summary (n={len(SEEDS_EXTENDED)}) ---")
print(f"  Baseline convergence: {b_converged}/{len(SEEDS_EXTENDED)} ({b_converged/len(SEEDS_EXTENDED)*100:.1f}%)")
print(f"  Attention convergence: {a_converged}/{len(SEEDS_EXTENDED)} ({a_converged/len(SEEDS_EXTENDED)*100:.1f}%)")
print(f"  Baseline R²: {np.mean(b_r2s):.4f} ± {np.std(b_r2s):.4f} (range [{min(b_r2s):.1f}, {max(b_r2s):.4f}])")
print(f"  Attention R²: {np.mean(a_r2s):.4f} ± {np.std(a_r2s):.4f} (range [{min(a_r2s):.4f}, {max(a_r2s):.4f}])")

all_results['extended_seeds'] = extended_seed_results
print("✅ Extended seed analysis complete")

## SAVE ALL RESULTS

In [ ]:
print("\n" + "="*70)
print("SAVING ALL RESULTS")
print("="*70)

# Save comprehensive results as JSON
with open(tables_dir / 'reviewer_experiment_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print("✅ All experiment results saved to tables/reviewer_experiment_results.json")
print("\nExperiment files created:")
print("  - baseline_comparison_full.csv")
print("  - attention_perturbation_results.csv")
print("  - hyperparameter_sensitivity.csv")
print("  - inference_latency.csv")
print("  - window_sensitivity.csv")
print("  - extended_seed_baseline.csv")
print("  - extended_seed_attention.csv")
print("  - reviewer_experiment_results.json")
print("  - y_test_mean_mc_dropout_attention.npy")
print("  - y_test_std_mc_dropout_attention.npy")